# RecruitGen — Model 1 of 3: Autoencoder (Resume Image Denoising)

**Goal:** remove noise from scanned / low-quality resume images so the text becomes reliably readable.

```
Noisy Image → Encoder → Latent Space → Decoder → Denoised Resume
```

**Training concept:** input = noisy resume, target = clean resume.
**Metrics:** MSE, PSNR, SSIM, visual comparison. **Regularisation:** Early Stopping.

**Outputs:** `encoder.h5` and `decoder.h5` (used by the Streamlit app).

## 1. Setup

In [ ]:
import os, glob, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import keras
from keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 256          # model works on 256x256 grayscale tiles
QUICK_RUN = False       # True = tiny run to test the notebook end-to-end
NUM_PAGES = 12 if QUICK_RUN else 160
PATCHES_PER_PAGE = 4 if QUICK_RUN else 24
EPOCHS = 1 if QUICK_RUN else 60
BATCH_SIZE = 16

DATA_DIR = "data/clean_resumes"   # optional: put your own clean resume images here
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## 2. Dataset — clean resume pages + synthetic noise

If `data/clean_resumes/` contains clean resume/document images they are used directly.
Otherwise realistic resume-like pages are rendered (name, headings, bullet points, two-column blocks).
Noisy versions are created with gaussian noise, salt & pepper, coffee stains, blur and scan shadows.

In [ ]:
WORDS = ("python java sql tensorflow pytorch aws azure docker kubernetes data analysis machine learning "
         "communication leadership project management engineer developer analyst cloud deployment "
         "university bachelor master certification experience responsible designed implemented improved "
         "pipeline dashboard model accuracy customers team agile api backend frontend testing").split()
HEADINGS = ["PROFESSIONAL SUMMARY", "EXPERIENCE", "EDUCATION", "SKILLS", "PROJECTS", "CERTIFICATIONS"]

def _font(size):
    for name in ["arial.ttf", "DejaVuSans.ttf", "LiberationSans-Regular.ttf"]:
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default()

def render_resume_page(w=1240, h=1754):
    img = Image.new("L", (w, h), 255)
    d = ImageDraw.Draw(img)
    y = 70
    d.text((80, y), " ".join(random.sample(WORDS, 2)).upper(), fill=0, font=_font(54)); y += 80
    d.text((80, y), " | ".join(random.sample(WORDS, 3)), fill=60, font=_font(24)); y += 60
    while y < h - 120:
        d.text((80, y), random.choice(HEADINGS), fill=0, font=_font(30)); y += 44
        d.line((80, y, w - 80, y), fill=90, width=2); y += 16
        for _ in range(random.randint(3, 7)):
            line = " ".join(random.choices(WORDS, k=random.randint(6, 12))).capitalize()
            d.text((100, y), "• " + line, fill=20, font=_font(random.choice([20, 22, 24]))); y += 34
        y += 24
    return img

def add_scan_noise(img):
    arr = np.asarray(img, dtype=np.float32) / 255.0
    h, w = arr.shape
    noisy = arr + np.random.normal(0, random.uniform(0.04, 0.15), arr.shape)
    m = np.random.rand(h, w)
    sp = random.uniform(0.005, 0.03)
    noisy[m < sp / 2] = 0.0
    noisy[m > 1 - sp / 2] = 1.0
    yy, xx = np.mgrid[0:h, 0:w]
    for _ in range(random.randint(1, 4)):          # stains
        cy, cx, r = random.randint(0, h), random.randint(0, w), random.randint(40, 260)
        noisy -= np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2)) * random.uniform(0.1, 0.35)
    noisy -= np.linspace(0, random.uniform(0, 0.25), w)[None, :]   # scan shadow
    out = Image.fromarray((np.clip(noisy, 0, 1) * 255).astype(np.uint8))
    if random.random() < 0.4:
        out = out.filter(ImageFilter.GaussianBlur(random.uniform(0.4, 1.0)))
    return out

def load_clean_pages():
    files = sorted(glob.glob(os.path.join(DATA_DIR, "*.*")))
    if files:
        print(f"Using {len(files)} clean images from {DATA_DIR}")
        return [Image.open(f).convert("L") for f in files]
    print(f"Rendering {NUM_PAGES} synthetic resume pages")
    return [render_resume_page() for _ in range(NUM_PAGES)]

def make_patches(pages):
    X, Y = [], []
    for page in pages:
        noisy = add_scan_noise(page)
        c, n = np.asarray(page, np.float32) / 255.0, np.asarray(noisy, np.float32) / 255.0
        h, w = c.shape
        for _ in range(PATCHES_PER_PAGE):
            y, x = random.randint(0, h - IMG_SIZE), random.randint(0, w - IMG_SIZE)
            X.append(n[y:y + IMG_SIZE, x:x + IMG_SIZE]); Y.append(c[y:y + IMG_SIZE, x:x + IMG_SIZE])
    return np.array(X)[..., None], np.array(Y)[..., None]

pages = load_clean_pages()
X, Y = make_patches(pages)
idx = np.random.permutation(len(X))
X, Y = X[idx], Y[idx]
n_val, n_test = int(0.15 * len(X)), int(0.10 * len(X))
X_test, Y_test = X[:n_test], Y[:n_test]
X_val, Y_val = X[n_test:n_test + n_val], Y[n_test:n_test + n_val]
X_train, Y_train = X[n_test + n_val:], Y[n_test + n_val:]
print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    ax[0, i].imshow(X_train[i, ..., 0], cmap="gray"); ax[0, i].set_title("Noisy input"); ax[0, i].axis("off")
    ax[1, i].imshow(Y_train[i, ..., 0], cmap="gray"); ax[1, i].set_title("Clean target"); ax[1, i].axis("off")
plt.tight_layout(); plt.show()

## 3. Model — convolutional Encoder → Latent Space → Decoder

U-Net style autoencoder: the encoder compresses the 256×256 tile into a **32×32×256 latent space**;
the decoder reconstructs the clean tile. Skip connections keep thin text strokes sharp.

In [ ]:
def conv(x, filters, name):
    return layers.Conv2D(filters, 3, padding="same", activation="relu", name=name)(x)

inp = keras.Input((IMG_SIZE, IMG_SIZE, 1), name="dirty_resume")
# ---------------- Encoder ----------------
e1 = conv(conv(inp, 32, "enc1_a"), 32, "enc1_b")                       # 256x256x32
e2 = conv(conv(layers.MaxPooling2D(name="pool1")(e1), 64, "enc2_a"), 64, "enc2_b")    # 128x128x64
e3 = conv(conv(layers.MaxPooling2D(name="pool2")(e2), 128, "enc3_a"), 128, "enc3_b")  # 64x64x128
latent = conv(layers.MaxPooling2D(name="pool3")(e3), 256, "latent_representation")  # 32x32x256
# ---------------- Decoder ----------------
d = conv(latent, 256, "dec_bottleneck")
d = layers.UpSampling2D(name="up1")(d); d = layers.Concatenate(name="skip1")([d, e3])
d = conv(conv(d, 128, "dec1_a"), 128, "dec1_b")
d = layers.UpSampling2D(name="up2")(d); d = layers.Concatenate(name="skip2")([d, e2])
d = conv(conv(d, 64, "dec2_a"), 64, "dec2_b")
d = layers.UpSampling2D(name="up3")(d); d = layers.Concatenate(name="skip3")([d, e1])
d = conv(conv(d, 32, "dec3_a"), 32, "dec3_b")
out = layers.Conv2D(1, 3, padding="same", activation="sigmoid", name="clean_resume")(d)

autoencoder = keras.Model(inp, out, name="Resume_Denoising_Autoencoder")

def psnr_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

def ssim_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=[psnr_metric, ssim_metric])
autoencoder.summary()

## 4. Training (with Early Stopping)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "ae_best.keras"), monitor="val_loss", save_best_only=True),
]
history = autoencoder.fit(
    X_train, Y_train, validation_data=(X_val, Y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, shuffle=True,
)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, key, title in zip(ax, ["loss", "psnr_metric", "ssim_metric"], ["MSE loss", "PSNR (dB)", "SSIM"]):
    a.plot(history.history[key], label="train"); a.plot(history.history["val_" + key], label="validation")
    a.set_title(title); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Evaluation — MSE · PSNR · SSIM on the test set

In [ ]:
pred = autoencoder.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

def report(name, a, b):
    mse = float(np.mean((a - b) ** 2))
    psnr = float(tf.reduce_mean(tf.image.psnr(a, b, 1.0)))
    ssim = float(tf.reduce_mean(tf.image.ssim(a, b, 1.0)))
    print(f"{name:22s} MSE={mse:.5f}  PSNR={psnr:6.2f} dB  SSIM={ssim:.4f}")

report("Noisy vs clean", X_test, Y_test)
report("Denoised vs clean", pred, Y_test)

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(14, 10))
for i in range(4):
    for r, (img, t) in enumerate([(X_test[i], "Noisy"), (pred[i], "Denoised"), (Y_test[i], "Clean")]):
        ax[r, i].imshow(img[..., 0], cmap="gray", vmin=0, vmax=1); ax[r, i].set_title(t); ax[r, i].axis("off")
plt.tight_layout(); plt.show()

## 6. Export — `encoder.h5` + `decoder.h5`

The encoder outputs the latent space plus the three skip feature maps; the decoder consumes them.
`decoder(encoder(x))` is identical to the full autoencoder.

In [ ]:
g = autoencoder.get_layer
encoder = keras.Model(inp, [latent, e3, e2, e1], name="resume_encoder")

lat_in = keras.Input((32, 32, 256), name="latent")
s3 = keras.Input((64, 64, 128), name="skip_64")
s2 = keras.Input((128, 128, 64), name="skip_128")
s1 = keras.Input((256, 256, 32), name="skip_256")
x = g("dec_bottleneck")(lat_in)
x = g("skip1")([g("up1")(x), s3]); x = g("dec1_b")(g("dec1_a")(x))
x = g("skip2")([g("up2")(x), s2]); x = g("dec2_b")(g("dec2_a")(x))
x = g("skip3")([g("up3")(x), s1]); x = g("dec3_b")(g("dec3_a")(x))
decoder = keras.Model([lat_in, s3, s2, s1], g("clean_resume")(x), name="resume_decoder")

encoder.save(os.path.join(OUTPUT_DIR, "encoder.h5"))
decoder.save(os.path.join(OUTPUT_DIR, "decoder.h5"))

check = decoder.predict(encoder.predict(X_test[:2], verbose=0), verbose=0)
print("max |decoder(encoder(x)) - autoencoder(x)| =", float(np.abs(check - autoencoder.predict(X_test[:2], verbose=0)).max()))
print("Saved:", os.listdir(OUTPUT_DIR))

Copy `outputs/encoder.h5` and `outputs/decoder.h5` into `models/autoencoder/` to use them in the app.